# The Data

## Background

Lithium-ion batteries dominate portable electronics and electric vehicles because lithium possesses a unique combination of physical and electrochemical properties. Its place as the lightest metal and the smallest metallic ion gives an exceptionally high energy density. In addition, lithium has the most negative standard electrode potential of any metal, allowing individual cells to operate at approximately 3.7 V—substantially higher than older rechargeable chemistries such as nickel-metal hydride or lead-acid batteries. Although lithium is less abundant than many other metals, these advantages have made lithium-ion technology the preferred choice for applications where high energy density and long cycle life are critical.

Lithium-ion batteries fail through repeated cycling, which gradually changes the battery's internal chemistry. The growth of the *Solid Electrolyte Interphase* (SEI), irreversible loss of active lithium, degradation of the electrode materials  and increasing internal resistance, progressively reduce the battery's ability to store and deliver energy. These processes subtly alter the battery's voltage and temperature response during charging and discharging, providing measurable signatures of ageing that can be exploited for health monitoring and prognostics.

The goal of this project is to investigate whether the first few minutes of a lithium-ion battery's discharge contain sufficient information to estimate its current health and future behaviour, while demonstrating a complete machine learning workflow including feature engineering, rigorous validation and model evaluation.

## Data Extraction

<span style="color:blue">**Because parsing the MATLAB files takes several minutes, the extracted data can optionally be saved as a CSV and loaded directly by Notebook 2.**</span>

### Exploring the MATLAB structure

The data are in MATLAB format and so the first stage is to convert the nested MATLAB structures into a tidy tabular format suitable for exploratory analysis and machine learning.

To run the notebook using the default dataset to see a full workflow without interaction, set

    USE_DEFAULTS = False

Change to

    USE_DEFAULTS = True

for full interaction.

In [1]:
from pathlib import Path
import sys
import os
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
project_root_abs = os.path.abspath(PROJECT_ROOT)
sys.path.insert(0, project_root_abs)

from src.loader import load_oxford_dataset

USE_DEFAULTS = True # True = runs presets silently | False = uses custom options below

In [2]:
from scipy.io import loadmat
import requests

DATA_FILE = Path("../data/Oxford_Battery_Degradation_Dataset_1.mat")

if not DATA_FILE.exists():
    print("Downloading Oxford Battery Degradation Dataset...")
    url = "https://ora.ox.ac.uk/objects/uuid:03ba4b01-cfed-46d3-9b1a-7d4a7bdf6fac/files/mc247ds58f"

    r = requests.get(url)
    r.raise_for_status()

    DATA_FILE.write_bytes(r.content)

mat_data = loadmat(DATA_FILE)

#infile = "../data/Oxford_Battery_Degradation_Dataset_1.mat"
#mat_data = loadmat(infile); display(mat_data.keys())

print('-- 8 battery cells - each one is almost certainly a MATLAB structure containing all measurements for that battery --\n')

cell = mat_data['Cell1']
cycle = cell['cyc0200'][0,0]

tests = cycle.dtype.names; print(tests)
print('-- 4 different tests --\n')

cycle = cell['cyc0100'][0,0] # select one diagnostic cycle.
test = cycle['C1dc'][0,0] # select one diagnostic test within that cycle
for field in test.dtype.names:
    arr = test[field][0,0].flatten()
    print(f"{field:>2} {arr.shape} {arr[:5]}")

print('-- Every measurement contains four synchronised vectors of equal length --') 

-- 8 battery cells - each one is almost certainly a MATLAB structure containing all measurements for that battery --

('C1ch', 'C1dc', 'OCVch', 'OCVdc')
-- 4 different tests --

 t (3565,) [735961.98322343 735961.98323501 735961.98324658 735961.98325816
 735961.98326973]
 v (3565,) [4.1926785 4.1765175 4.1749935 4.1737504 4.1725473]
 q (3565,) [ 0.         -0.19481253 -0.39974963 -0.60468765 -0.80962484]
 T (3565,) [39.964054 39.8517   39.876675 39.914112 39.826733]
-- Every measurement contains four synchronised vectors of equal length --


Preliminary testing, shows the hierarchy
 
     Cell 1
       │
       ├── Cycle 0000
       │      │
       │      ├── C1ch
       │      ├── C1dc
       │      ├── OCVch
       │      └── OCVdc
       │
       ├── Cycle 0100
       │      │
       │      └── C1dc
       │             │
       │             ├── t (3544)
       │             ├── v (3544)
       │             ├── q (3544)
       │             └── T (3544)
       
       
Where eight cells are tested over up to $\approx80$ cycles, each spanning up to 
$\approx60$ minutes duration. 

Four separate diagnostic tests are performed:

 - **C1ch**:    1C charge test
 - **C1dc**:    1C discharge test
 - **OCVch**:   Open Circuit Voltage charge test
 - **OCVdc**    Open Circuit Voltage discharge test

each containing the voltage (V), the discharged capacity (mAh) and the temperature (<sup>o</sup>C) over the duration.
 
It is noted that: 

- The voltage in the cells range from 4.2 to 2.7 V, which is consistent with the fully charged and discharged values for lithium-ion cells.
- The time units are MATLAB datenums, which is the number of days in respect to a reference data. We convert these to minutes since one hour is consistent with a 1 C discharge.

### Constructing the dataframe

In the notebook each test can be selected by toggling off the default mode, via

	USE_DEFAULTS = False

which will also give the option to save to a file.

In [3]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Global Variable Assignments
df = None

if USE_DEFAULTS:
    # UPDATED: Path perfectly mirrors your widget defaults ('C1dc' and 'csv')
    default_csv = "../data/../data/Oxford_Battery_Dataset_Test=C1dc.csv"
    print(f"⚙️ Running Mode: [DEFAULTS ACTIVE] | Target: {default_csv}")
    
    if os.path.exists(default_csv):
        print(f"📦 Reading pre-compiled file from disk...")
        df = pd.read_csv(default_csv)
        print(f"✅ Success! Loaded {len(df):,} rows into memory.")
        
    else:
        print(f"❌ Error: Default file not found at {default_csv}.")
        print("💡 Set USE_DEFAULTS = False, choose 'C1dc', set 'Save data' to 'csv', and click 'Load Data' to create it.")

else:
    dd_test = widgets.Dropdown(
        options=['C1ch', 'C1dc', 'OCVch', 'OCVdc'], 
        value='C1dc', description='Test:', layout={'width': '25%'}
    )
    dd_save = widgets.Dropdown(
        options=['No', 'csv', 'parquet'], 
        value='No', description='Save data:', layout={'width': '25%'}
    )
    
    ui = widgets.HBox([dd_test, dd_save])
    menu = widgets.Output()

    def handle_dropdown_change(change):
        global df
        with menu:
            clear_output(wait=True)
            test_type = dd_test.value
            save_format = dd_save.value
            
            print(f"🔄 Reloading dataset... Test Target: {test_type} | Save Format: {save_format}")
            df = load_oxford_dataset(test_type)
            
            # Process save operations based on selections
            if save_format != 'No':
                out_base = f"../data/Oxford_Battery_Dataset_test={test_type}"
                os.makedirs("../data/", exist_ok=True)
                
                if save_format == 'csv':
                    df.to_csv(f"{out_base}.csv", index=False)
                    print(f"💾 File successfully written to: {out_base}.csv")
                elif save_format == 'parquet':
                    df.to_parquet(f"{out_base}.parquet", index=False)
                    print(f"💾 File successfully written to: {out_base}.parquet")
                    
            print(f"✅ Success! Loaded {len(df):,} total data rows into global variable 'df'.")
            print("---- Loaded Oxford dataset ----")
            print("Batteries: %d" %(df.Cell.nunique())) 
            print("Cycles   : %d" %(df.Cycle.nunique())) 
            print("Rows    : %d" %(len(df)))
            print("Test    : %s" %(df.Test.unique()))      

    
    dd_test.observe(handle_dropdown_change, names='value')
    dd_save.observe(handle_dropdown_change, names='value')
    display(ui, menu)

Output()